# IIR Filtering from Pseudocode to Python

A reproducible translation of a language-agnostic algorithm into NumPy and SciPy.

## Audience, prerequisites, and learning goals

**Audience:** learners who know basic Python arrays and want to connect algorithm design with signal-processing code.

**Prerequisites:** zero-indexed arrays, the canonical IQ layout `(N, 2, L)`, and average signal power.

You will learn to:

1. Specify an IIR filtering pipeline in disciplined pseudocode.
2. Translate each function directly to Python.
3. Apply a second-order Butterworth low-pass filter along the time axis.
4. Validate shape, power, and numerical behavior.

## Data and filter contract

- `N = 5` examples.
- `2` components: index `0` is I and index `1` is Q.
- `L = 1000` time samples.
- `SEED = 42`.
- The normalized cutoff is `0.1`, where `1.0` represents the Nyquist frequency.
- Arrays are zero-indexed.

## 1. Algorithm in pseudocode

```pseudocode
FUNCTION GenerateSyntheticIQ(numExamples AS INTEGER, numSamples AS INTEGER, seed AS INTEGER) RETURNS ARRAY
    DECLARE randomGenerator AS RANDOM_GENERATOR
    DECLARE timeIndex AS ARRAY
    DECLARE iTone AS ARRAY
    DECLARE qTone AS ARRAY
    DECLARE lowFrequencyTone AS ARRAY
    DECLARE highFrequencyNoise AS ARRAY
    DECLARE signal AS ARRAY

    randomGenerator ← CreateRandomGenerator(seed)
    timeIndex ← Range(0, numSamples - 1)
    iTone ← Cos(2 * PI * 0.02 * timeIndex)
    qTone ← Sin(2 * PI * 0.02 * timeIndex)
    lowFrequencyTone ← Stack([iTone, qTone], axis = 0)
    highFrequencyNoise ← 0.35 * randomGenerator.StandardNormal([numExamples, 2, numSamples])
    signal ← Repeat(lowFrequencyTone, shape = [numExamples, 2, numSamples]) + highFrequencyNoise

    ASSERT Shape(signal) = [numExamples, 2, numSamples]
    RETURN signal
END FUNCTION

FUNCTION DesignButterworthIIR(cutoff AS FLOAT, order AS INTEGER) RETURNS TUPLE
    DECLARE numerator AS ARRAY
    DECLARE denominator AS ARRAY

    (numerator, denominator) ← ButterworthLowPass(order, cutoff)
    RETURN (numerator, denominator)
END FUNCTION

FUNCTION ApplyIIRFilter(signal AS ARRAY, numerator AS ARRAY, denominator AS ARRAY) RETURNS ARRAY
    DECLARE filtered AS ARRAY

    filtered ← LinearFilter(numerator, denominator, signal, axis = 2)
    RETURN filtered
END FUNCTION

FUNCTION ComputePower(iqArray AS ARRAY) RETURNS FLOAT
    DECLARE iComponent AS ARRAY
    DECLARE qComponent AS ARRAY

    iComponent ← iqArray[:, 0, :]
    qComponent ← iqArray[:, 1, :]
    RETURN Mean(iComponent ^ 2 + qComponent ^ 2)
END FUNCTION

ALGORITHM FilterSyntheticIQ
    DECLARE signal AS ARRAY
    DECLARE filteredSignal AS ARRAY
    DECLARE numerator AS ARRAY
    DECLARE denominator AS ARRAY
    DECLARE powerBefore AS FLOAT
    DECLARE powerAfter AS FLOAT

    signal ← GenerateSyntheticIQ(5, 1000, 42)
    (numerator, denominator) ← DesignButterworthIIR(0.1, 2)
    filteredSignal ← ApplyIIRFilter(signal, numerator, denominator)
    powerBefore ← ComputePower(signal)
    powerAfter ← ComputePower(filteredSignal)

    ASSERT Shape(filteredSignal) = Shape(signal)
    ASSERT powerAfter ≠ powerBefore
    PRINT powerBefore
    PRINT powerAfter
END ALGORITHM
```

## 2. Translation map

| Pseudocode | Python/SciPy |
| --- | --- |
| `CreateRandomGenerator(seed)` | `np.random.default_rng(seed)` |
| `ButterworthLowPass(order, cutoff)` | `scipy.signal.butter(order, cutoff, btype="lowpass")` |
| `LinearFilter(..., axis = 2)` | `scipy.signal.lfilter(..., axis=2)` |
| `Mean(I ^ 2 + Q ^ 2)` | `np.mean(I**2 + Q**2)` |

In [ ]:
import numpy as np
from scipy import signal as scipy_signal

In [ ]:
def generate_synthetic_iq(num_examples, num_samples, seed):
    rng = np.random.default_rng(seed)
    time_index = np.arange(num_samples, dtype=np.float32)

    phase = 2 * np.pi * 0.02 * time_index
    low_frequency_tone = np.stack((np.cos(phase), np.sin(phase)))
    low_frequency_tone = np.broadcast_to(
        low_frequency_tone, (num_examples, 2, num_samples)
    )

    high_frequency_noise = 0.35 * rng.standard_normal(
        (num_examples, 2, num_samples)
    )
    iq_signal = (low_frequency_tone + high_frequency_noise).astype(np.float32)

    assert iq_signal.shape == (num_examples, 2, num_samples)
    return iq_signal

In [ ]:
def design_butterworth_iir(cutoff, order):
    return scipy_signal.butter(order, cutoff, btype="lowpass")


def apply_iir_filter(iq_signal, numerator, denominator):
    return scipy_signal.lfilter(
        numerator, denominator, iq_signal, axis=2
    ).astype(np.float32)


def compute_power(iq_array):
    i_component = iq_array[:, 0, :]
    q_component = iq_array[:, 1, :]
    return float(np.mean(i_component**2 + q_component**2))

## 3. Execute the translated algorithm

The high-frequency noise should be attenuated, so average power is expected to decrease while shape remains unchanged.

In [ ]:
N = 5
L = 1_000
SEED = 42
CUTOFF = 0.1
ORDER = 2

X = generate_synthetic_iq(N, L, SEED)
b, a = design_butterworth_iir(CUTOFF, ORDER)
X_filtered = apply_iir_filter(X, b, a)

power_before = compute_power(X)
power_after = compute_power(X_filtered)
power_ratio = power_after / power_before

print(f"Input shape:  {X.shape}")
print(f"Output shape: {X_filtered.shape}")
print(f"Numerator coefficients:   {b}")
print(f"Denominator coefficients: {a}")
print(f"Power before: {power_before:.6f}")
print(f"Power after:  {power_after:.6f}")
print(f"Power ratio:  {power_ratio:.6f}")

## 4. Validation

These checks prove that filtering occurs on the time axis without changing the canonical IQ layout.

In [ ]:
assert X.shape == (N, 2, L)
assert X_filtered.shape == X.shape
assert X.dtype == np.float32
assert X_filtered.dtype == np.float32
assert np.isfinite(X_filtered).all()
assert not np.allclose(X_filtered, X)
assert power_after < power_before
assert 0.0 < power_ratio < 1.0

print("PASS: shape, dtype, finiteness, filtering, and power change verified.")

## Exercise

Change the cutoff from `0.1` to `0.3`. Before running the filter again, predict whether more or less high-frequency content will pass and how the power ratio will change.

**Answer scaffold:** Increasing the cutoff creates a ______ passband, so the output retains ______ energy and the power ratio should ______.

In [ ]:
# Optional exercise
exercise_cutoff = 0.3
# b_exercise, a_exercise = design_butterworth_iir(exercise_cutoff, ORDER)
# X_exercise = apply_iir_filter(X, b_exercise, a_exercise)
# print(f"Exercise power ratio: {compute_power(X_exercise) / power_before:.6f}")

## Common pitfalls and extensions

- `axis=2` is essential: it identifies time in `(N, 2, L)`. Filtering axis 1 would mix the I/Q component dimension instead.
- `cutoff=0.1` is normalized to Nyquist because no sampling frequency is supplied.
- `lfilter` starts with zero initial conditions, so the beginning contains a transient.

**Extension:** compare `lfilter` with zero-phase `sosfiltfilt`, but explain why zero-phase processing is non-causal.